In [5]:
from googleapiclient.discovery import build
import pandas as pd

# -----------------------------
# 1) Set your API key here
# -----------------------------
API_KEY = "REDACTED_YOUTUBE_API_KEY"

# Build YouTube client
youtube = build("youtube", "v3", developerKey=API_KEY)


# -----------------------------
# 2) Function to get all playlists
# -----------------------------
def get_all_channel_playlists(youtube, channel_id):
    """
    Extract all public playlists from a YouTube channel.

    Parameters
    ----------
    youtube : googleapiclient.discovery.Resource
        YouTube API client.
    channel_id : str
        YouTube channel ID.

    Returns
    -------
    pd.DataFrame
        Data frame with one row per playlist.
    """
    playlists = []
    next_page = None

    while True:
        resp = youtube.playlists().list(
            part="id,snippet,contentDetails,status",
            channelId=channel_id,
            maxResults=50,
            pageToken=next_page
        ).execute()

        items = resp.get("items", [])

        for item in items:
            playlists.append({
                "channel_id": channel_id,
                "playlist_id": item.get("id"),
                "playlist_title": item["snippet"].get("title"),
                "playlist_description": item["snippet"].get("description"),
                "published_at": item["snippet"].get("publishedAt"),
                "item_count": item["contentDetails"].get("itemCount"),
                "privacy_status": item["status"].get("privacyStatus"),
            })

        next_page = resp.get("nextPageToken")
        if not next_page:
            break

    return pd.DataFrame(playlists)


# -----------------------------
# 3) Example: Milenio
# -----------------------------
CHANNEL_ID = "UCFxHplbcoJK9m70c4VyTIxg"  # replace if needed

df_playlists = get_all_channel_playlists(youtube, CHANNEL_ID)

print(df_playlists.head())
print(f"Total playlists: {len(df_playlists)}")


# -----------------------------
# 4) Save to CSV
# -----------------------------
df_playlists.to_csv("milenio_playlists.csv", index=False)

                 channel_id                         playlist_id  \
0  UCFxHplbcoJK9m70c4VyTIxg  PLRzDietc_2KhUYj1gGtpKLYBzHtx-8pJx   
1  UCFxHplbcoJK9m70c4VyTIxg  PLRzDietc_2Kj9gAjNMv0PkKlHhIjEwGca   
2  UCFxHplbcoJK9m70c4VyTIxg  PLRzDietc_2KjFpPNF8t5-HTdNsxiXqHqi   
3  UCFxHplbcoJK9m70c4VyTIxg  PLRzDietc_2KhtRbJHwcIX0teahYVWdqlI   
4  UCFxHplbcoJK9m70c4VyTIxg  PLRzDietc_2Kije-pIoNoJno_FrKEz_ABO   

            playlist_title                               playlist_description  \
0      Premios Oscars 2026                                                      
1          Eventos en vivo  Discursos, inauguraciones y declaraciones de C...   
2                8M - 2026                                                      
3  La caída de 'El Mencho'  Cobertura completa sobre la caída del líder ab...   
4                     Cuba                  Noticias de actualidad sobre Cuba   

                  published_at  item_count privacy_status  
0  2026-03-13T13:26:57.609264Z           9        

In [4]:
resp = youtube.search().list(
    part="snippet",
    q="milenio",
    type="channel",
    maxResults=5
).execute()

for item in resp["items"]:
    print(item["snippet"]["title"], item["snippet"]["channelId"])

MILENIO UCFxHplbcoJK9m70c4VyTIxg
Milenio 3 - La nave del misterio UCVV-yK-iSFrdD3sKyPXvMsg
Milenio Milenio UC6GGMUHQmUTu_kGaPiQSeBw
Milenio 3 - En el Ártico UCBMnEsGyEZmCUUvJGMC4hMQ
Nuevo Milenio UCBrScPKHXYwTKflB5LFVJhA
